# Go2 Track 2 Bonus Project: Colab Starter Notebook

Use this notebook to set up the repo, inspect the interfaces, optionally train or reuse a low-level checkpoint, run single-policy track evaluation, and prepare submission metadata.

It starts from a weak baseline. Your leaderboard submission should train a learned high-level planner for the fixed 5D -> [vx, vy, yaw_rate] interface.


## 1. Configure repository URLs

Leave the default repo URL unless you are working from your own fork. If you rerun setup after editing files, keep `RESET_COURSE_REPO = False` so your local changes are not deleted.


In [1]:
from pathlib import Path
import io
import os
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.request
from urllib.parse import urlparse

COURSE_REPO_URL = "https://github.com/yuh-w/Final-Project-Track-2-Bonus-Project.git"
COURSE_REPO_BRANCH = "main"
TEAM_NAME = "change_me"
RESET_COURSE_REPO = False
BASE_DIR = Path("/content") if Path("/content").exists() else Path("/tmp/go2_track_bonus_colab")
BASE_DIR.mkdir(parents=True, exist_ok=True)
COURSE_REPO_DIR = BASE_DIR / "go2_track_bonus_repo"

PLAYGROUND_REPO = "https://github.com/google-deepmind/mujoco_playground.git"
PLAYGROUND_REF = "dd38c285c6d54266287081e516109f0b15985818"

UNITREE_MUJOCO_REPO = "https://github.com/unitreerobotics/unitree_mujoco.git"
UNITREE_MUJOCO_REF = "1a37b051a10be723405b7ed6dc839361af036d88"

MENAGERIE_REPO = "https://github.com/deepmind/mujoco_menagerie.git"
MENAGERIE_REF = "1b86ece576591213e2b666ebf59508454200ca97"

PLAYGROUND_DIR = BASE_DIR / "mujoco_playground"
UNITREE_DIR = BASE_DIR / "unitree_mujoco"
MENAGERIE_DIR = PLAYGROUND_DIR / "mujoco_playground" / "external_deps" / "mujoco_menagerie"

def run(cmd):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True)

def github_archive_url(repo_url: str, ref: str) -> str:
    repo_path = urlparse(repo_url).path.strip("/")
    if repo_path.endswith(".git"):
        repo_path = repo_path[:-4]
    return f"https://codeload.github.com/{repo_path}/tar.gz/{ref}"

def download_repo_snapshot(repo_url: str, ref: str, target_dir: Path) -> None:
    archive_url = github_archive_url(repo_url, ref)
    print(f"+ download {archive_url}")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = Path(tempfile.mkdtemp(prefix=f"{target_dir.name}_", dir=str(target_dir.parent)))
    try:
        with urllib.request.urlopen(archive_url) as response:
            payload = response.read()
        with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
            archive.extractall(tmp_dir)
        extracted_dirs = [path for path in tmp_dir.iterdir() if path.is_dir()]
        if len(extracted_dirs) != 1:
            raise RuntimeError(f"Expected one extracted directory, got {extracted_dirs}")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.move(str(extracted_dirs[0]), str(target_dir))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

def checkout_existing_repo(target_dir: Path, ref: str) -> None:
    try:
        run(["git", "-C", target_dir, "fetch", "--all", "--tags"])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git fetch failed for {target_dir}: {exc}. Trying local checkout.")
    run(["git", "-C", target_dir, "checkout", ref])

def ensure_pinned_repo(repo_url: str, ref: str, target_dir: Path) -> None:
    if target_dir.exists() and (target_dir / ".git").exists():
        try:
            checkout_existing_repo(target_dir, ref)
            return
        except subprocess.CalledProcessError as exc:
            print(f"[warn] local git checkout failed for {target_dir}: {exc}. Re-downloading snapshot.")
            shutil.rmtree(target_dir)
    elif target_dir.exists():
        shutil.rmtree(target_dir)

    try:
        run(["git", "clone", repo_url, target_dir])
        checkout_existing_repo(target_dir, ref)
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git path failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, ref, target_dir)

def ensure_course_repo(repo_url: str, branch: str, target_dir: Path) -> None:
    if target_dir.exists():
        if RESET_COURSE_REPO:
            print(f"+ remove existing course repo at {target_dir}")
            shutil.rmtree(target_dir)
        else:
            print(f"+ reuse existing course repo at {target_dir}")
            return
    try:
        run(["git", "clone", repo_url, target_dir])
    except subprocess.CalledProcessError as exc:
        print(f"[warn] git clone failed for {repo_url}: {exc}. Falling back to archive download.")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, branch, target_dir)

if "google.colab" in sys.modules:
    print("Running inside Colab.")
else:
    print("This notebook was designed for Colab, but local execution may also work.")


Running inside Colab.


## 2. Install system packages and clone repositories


In [2]:
import shutil
if shutil.which("ffmpeg") is None:
    if Path("/content").exists():
        run(["apt-get", "update", "-qq"])
        run(["apt-get", "install", "-y", "ffmpeg"])
    else:
        print("[local] ffmpeg is not on PATH; imageio-ffmpeg from requirements is enough for local tests.")
!python -m pip install -q -U pip setuptools wheel
!python -m pip uninstall -y playground || true

ensure_pinned_repo(PLAYGROUND_REPO, PLAYGROUND_REF, PLAYGROUND_DIR)
ensure_pinned_repo(UNITREE_MUJOCO_REPO, UNITREE_MUJOCO_REF, UNITREE_DIR)
ensure_course_repo(COURSE_REPO_URL, COURSE_REPO_BRANCH, COURSE_REPO_DIR)
ensure_pinned_repo(MENAGERIE_REPO, MENAGERIE_REF, MENAGERIE_DIR)

!python -m pip install -q -r {COURSE_REPO_DIR / 'configs' / 'colab_requirements.txt'}
%cd {PLAYGROUND_DIR}
!python -m pip install -q -e .
%cd {COURSE_REPO_DIR}

import sys
if str(PLAYGROUND_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(PLAYGROUND_DIR.resolve()))

import jax
import mujoco_playground

print("JAX devices:", jax.devices())
print("JAX backend:", jax.default_backend())
print("mujoco_playground imported from:", mujoco_playground.__file__)
expected_playground = str(PLAYGROUND_DIR.resolve())
if expected_playground not in str(Path(mujoco_playground.__file__).resolve()):
    raise RuntimeError(f"Expected mujoco_playground to be imported from {expected_playground}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.
+ git clone https://github.com/google-deepmind/mujoco_playground.git /content/mujoco_playground
+ git -C /content/mujoco_playground fetch --all --tags
+ git -C /content/mujoco_playground checkout dd38c285c6d54266287081e516109f0b15985818
+ git clone https://github.com/unitreerobotics/unitree_mujoco.git /content/unitree_mujoco
+ git -C /content/unitree_mujoco fetch --all --tags
+ git -C /content/unitree_mujoco checkout 1a37b051a10be723405b7ed6dc839361af036d88
+ git clone https://github.com/yuh-w/Final-Project-T

## 3. Read the assignment requirements

Skim the short specs before editing.


In [ ]:
%cd {COURSE_REPO_DIR}
!sed -n '1,180p' docs/assignment_requirements.md
!sed -n '1,140p' docs/controller_interface.md
!sed -n '1,120p' docs/high_level_optimization_guide.md


/content/go2_track_bonus_repo
# Track 2 Requirements

Goal: run Go2 as far as possible around a 200 m oval track in MuJoCo.

## Options

- Proposal-based final project.
- Go2 oval-track leaderboard route.
- Both, for bonus.

## Leaderboard Route

```text
5D track observation -> [vx, vy, yaw_rate] -> Go2 low-level policy
```

Use `docs/controller_interface.md`. The low-level checkpoint should stay
compatible with the HW1 Brax PPO format.
This repo evaluates one submission at a time; ranking compares submitted
outputs.

Leaderboard submissions must train a learned high-level planner for this fixed
interface. The provided starter planner is only a weak baseline for debugging.
Keep the 5D input and 3D output fixed; change the planner internals. The
official scene is fixed: 200 m centerline, 18.25 m turn radius, and 2.0 m half
width. Do not change the track geometry to improve score.

## Allowed

- Reuse a HW1 checkpoint.
- Retrain or modify the low-level Go2 policy.
- Train a learned high-

## 4. Copy Go2 assets


In [3]:
%cd {COURSE_REPO_DIR}
!python scripts/copy_go2_assets.py --unitree-dir {UNITREE_DIR} --course-dir {COURSE_REPO_DIR}


/content/go2_track_bonus_repo
Copied 16 assets into /content/go2_track_bonus_repo/go2_pg_env/xmls/assets


## 5. Inspect the low-level Go2 environment


In [ ]:
%cd {COURSE_REPO_DIR}
!python inspect_env.py --stage-name stage_2


/content/go2_track_bonus_repo
Traceback (most recent call last):
  File "/content/go2_track_bonus_repo/inspect_env.py", line 109, in <module>
    main()
  File "/content/go2_track_bonus_repo/inspect_env.py", line 66, in main
    stack = lazy_import_stack()
            ^^^^^^^^^^^^^^^^^^^
  File "/content/go2_track_bonus_repo/course_common.py", line 107, in lazy_import_stack
    register_go2_env()
  File "/content/go2_track_bonus_repo/go2_pg_env/__init__.py", line 16, in register
    from . import randomize
  File "/content/go2_track_bonus_repo/go2_pg_env/randomize.py", line 16, in <module>
    _MODEL = mujoco.MjModel.from_xml_path(consts.FEET_ONLY_FLAT_TERRAIN_XML.as_posix())
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: Error: Error opening file '/content/go2_track_bonus_repo/go2_pg_env/xmls/base_1.obj'


## 6. Read the important starter files


In [ ]:
!sed -n '1,180p' go2_pg_env/joystick.py
!sed -n '1,220p' go2_pg_env/track.py
!sed -n '1,220p' track_bonus/controller_interface.py
!sed -n '1,220p' track_bonus/planner.py
!sed -n '1,220p' run_track_bonus.py


"""Joystick locomotion task for the local Go2 environment.

This task is adapted from MuJoCo Playground's Go1 joystick task. The local
changes are intentionally small so that students can compare the official
baseline against a course-specific Go2 variant.

Observation summary
-------------------
state (actor input):
    [local_linvel(3), gyro(3), gravity(3),
     joint_pos_error(12), joint_vel(12),
     last_action(12), command(3)]  -> 48 dims

privileged_state (critic-only input during training):
    state + extra simulator-only signals -> 123 dims

Action summary
--------------
The policy outputs 12 joint offsets. The final motor target is:
    target_joint_pos = default_pose + action_scale * policy_action
"""

from __future__ import annotations

from typing import Any, Dict, Optional, Union

import jax
import jax.numpy as jp
from ml_collections import config_dict
from mujoco import mjx
from mujoco.mjx._src import math
import numpy as np

from mujoco_playground._src import mjx_env



## 7. Define a Colab-friendly low-level training config

This is a normal Colab training starting point, not a quick test. Reduce the step counts for experiments if needed.


In [ ]:
import json

runtime_config = {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [256, 256, 128],
    "value_hidden_layer_sizes": [256, 256, 128],
    "stage_1_num_timesteps": 10_000_000,
    "stage_2_num_timesteps": 5_000_000,
}

config_path = COURSE_REPO_DIR / "configs" / "colab_runtime_config.json"
base_config_path = COURSE_REPO_DIR / "configs" / "course_config.json"
base_config = json.loads(base_config_path.read_text())
base_config["runtime_overrides"] = runtime_config
config_path.write_text(json.dumps(base_config, indent=2))
print("wrote", config_path)


wrote /content/go2_track_bonus_repo/configs/colab_runtime_config.json


## 8. Dry-run training config


In [ ]:
!python train.py --config configs/colab_runtime_config.json --dry-run


## 9. Train or reuse a low-level checkpoint

Use a newly trained checkpoint or point `CHECKPOINT_DIR` to a HW1 `best_checkpoint`.


In [4]:
# Clear out the old corrupted folder
!rm -rf /content/go2_track_bonus_repo/artifacts/low_level_train/best_checkpoint

# Ensure the parent directory path exists
!mkdir -p /content/go2_track_bonus_repo/artifacts/low_level_train/

# Extract your clean checkpoint archive directly into the target directory  hw1_best_checkpoint/best_checkpoint-cbea1a
!unzip /content/go2_track_bonus_repo/best_checkpoint-cbea1a.zip -d /content/go2_track_bonus_repo/artifacts/low_level_train/

Archive:  /content/go2_track_bonus_repo/best_checkpoint-cbea1a.zip
  inflating: /content/go2_track_bonus_repo/artifacts/low_level_train/best_checkpoint/manifest.ocdbt  
  inflating: /content/go2_track_bonus_repo/artifacts/low_level_train/best_checkpoint/ocdbt.process_0/manifest.ocdbt  
  inflating: /content/go2_track_bonus_repo/artifacts/low_level_train/best_checkpoint/ocdbt.process_0/d/5a615e3d1928c4922823f19695a2a55a  
  inflating: /content/go2_track_bonus_repo/artifacts/low_level_train/best_checkpoint/manifest.json  
  inflating: /content/go2_track_bonus_repo/artifacts/low_level_train/best_checkpoint/_CHECKPOINT_METADATA  
  inflating: /content/go2_track_bonus_repo/artifacts/low_level_train/best_checkpoint/ppo_network_config.json  
  inflating: /content/go2_track_bonus_repo/artifacts/low_level_train/best_checkpoint/_METADATA  
  inflating: /content/go2_track_bonus_repo/artifacts/low_level_train/best_checkpoint/d/9c7a9b1ae76680f8e966afab27c1b5d3  
  inflating: /content/go2_track_bonu

In [5]:
# Clear out the old corrupted folder
!rm -rf /content/go2_track_bonus_repo/artifacts/highlevel_train/

# Ensure the parent directory path exists
!mkdir -p /content/go2_track_bonus_repo/artifacts/highlevel_train/

# Extract your clean checkpoint archive directly into the target directory  hw1_best_checkpoint/best_checkpoint-cbea1a
!unzip /content/go2_track_bonus_repo/final_submission.zip -d /content/go2_track_bonus_repo/artifacts/highlevel_train/

Archive:  /content/go2_track_bonus_repo/final_submission.zip
  inflating: /content/go2_track_bonus_repo/artifacts/highlevel_train/final_submission/submission.json  
  inflating: /content/go2_track_bonus_repo/artifacts/highlevel_train/final_submission/planner_config.json  
  inflating: /content/go2_track_bonus_repo/artifacts/highlevel_train/final_submission/planner_weights.npz  


In [ ]:
# Uncomment the command below to train a new low-level policy.
# !python train.py \
#   --config configs/colab_runtime_config.json \
#   --stage both \
#   --output-dir artifacts/low_level_train

# Or set CHECKPOINT_DIR to an existing HW1 best_checkpoint, for example:
# CHECKPOINT_DIR = Path("/content/path/to/your/HW1/best_checkpoint")
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "low_level_train" / "best_checkpoint"
PLANNER_CONFIG = COURSE_REPO_DIR / "configs" / "starter_planner.json"
print("checkpoint path:", CHECKPOINT_DIR)
print("planner config:", PLANNER_CONFIG)
print("checkpoint exists:", CHECKPOINT_DIR.exists())


checkpoint path: /content/go2_track_bonus_repo/artifacts/low_level_train/best_checkpoint
planner config: /content/go2_track_bonus_repo/configs/starter_planner.json
checkpoint exists: True


## 10. Run single-policy track evaluation

The smoke command is short and writes to `track_eval_smoke`. For submission, run the full command into `track_eval`.


In [ ]:
TRACK_EVAL_SMOKE_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval_smoke"
TRACK_EVAL_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval"

if CHECKPOINT_DIR.exists():
    print("Found checkpoint. Running short no-render track smoke eval...")
    !python run_track_bonus.py \
      --checkpoint-dir {CHECKPOINT_DIR} \
      --planner-config {PLANNER_CONFIG} \
      --config configs/colab_runtime_config.json \
      --output-dir {TRACK_EVAL_SMOKE_DIR} \
      --entry-name {TEAM_NAME} \
      --duration-seconds 5 \
      --no-render
else:
    print("Skipping track eval: CHECKPOINT_DIR does not exist yet.")
    print("Train a policy above or set CHECKPOINT_DIR to your HW1 best_checkpoint, then rerun this cell.")

# Full evaluation with video, after the smoke run works:
# !python run_track_bonus.py \
#   --checkpoint-dir {CHECKPOINT_DIR} \
#   --planner-config {PLANNER_CONFIG} \
#   --config configs/colab_runtime_config.json \
#   --output-dir {TRACK_EVAL_DIR} \
#   --entry-name {TEAM_NAME} \
#   --render-every 10 \
#   --render-fps 5


## 11. Train your high-level planner

You are expected to train a learned high-level planner. The command below is only a starter parameter search for debugging the loop; replace the planner internals with your own MLP, RL policy, or other trained policy while keeping the same 5D -> [vx, vy, yaw_rate] interface.

Do not change the track geometry. The evaluator uses the fixed official oval for reset, scoring, and rendering.

A valid learned planner has trained parameters, such as MLP weights. Store those weights in your submission and load them from `StarterTrackPlanner.load(planner_config)`.


In [16]:
!python sweep_highlevel_settings_colab.py \
  --checkpoint-dir artifacts/low_level_train/best_checkpoint \
  --settings robust_38,robust_36 \
  --iterations 4 \
  --population 4096 \
  --train-eval-seconds 76 \
  --official-eval-seconds 80 \
  --top-k-results 2 \
  --eval-top-k 3 \
  --eval-seeds 20260527, 20260528, 20260529


=== Training setting: robust_38 ===

$ /usr/bin/python3 train_mlp_cem.py --checkpoint-dir artifacts/low_level_train/best_checkpoint --config /content/go2_track_bonus_repo/configs/course_config.json --output-dir /content/go2_track_bonus_repo/artifacts/highlevel_sweeps/robust_38 --iterations 4 --population 4096 --elite-frac 0.2 --eval-seconds 76.0 --hidden-dim 32 --teacher-steps 1200 --teacher-samples 8192 --teacher-batch-size 512 --teacher-lr 0.003 --top-k-results 2 --seed 42 --command-filter-alpha 0.35 --max-straight-speed-mps 3.8 --max-curve-speed-mps 3.4 --max-lateral-speed-mps 0.22 --max-yaw-rate-radps 0.6 --edge-slowdown-margin-norm 0.15 --max-command-delta 0.15 --boundary-safety-margin-m 0.05
Initializing JAX and loading MuJoCo/Brax environment (This may take 1-2 minutes)...
Traceback (most recent call last):
  File "/usr/lib/python3.12/subprocess.py", line 1264, in wait
    return self._wait(timeout=timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/subp

In [ ]:
HIGHLEVEL_DIR = COURSE_REPO_DIR / "artifacts" / "highlevel_train"

# 1. Train the high level network weights using CEM
!python train_mlp_cem.py \
  --checkpoint-dir artifacts/low_level_train/best_checkpoint \
  --output-dir artifacts/highlevel_train \
  --iterations 8 \
  --population 4096 \
  --eval-seconds 78.0 \
  --teacher-steps 1200 \
  --teacher-samples 8192 \
  --teacher-lr 0.003

Initializing JAX and loading MuJoCo/Brax environment (This may take 1-2 minutes)...
Environment loaded and compiled! Starting VMAP CEM Optimization...

Distilling teacher planner into MLP warm start (1200 steps, 8192 samples)...
Teacher distillation warm start finished: loss=0.00225
[Gen 00] Best Gen Score: 70.308 | Best Global: 70.308 | Lap: 100.0% | Mean Speed: 2.56 m/s | Speed Score: 0.260 | Recent Speed: 2.71 m/s | Fall: False | Boundary: False | Step Time: 217.161s
[Gen 01] Best Gen Score: 77.802 | Best Global: 77.802 | Lap: 100.0% | Mean Speed: 2.56 m/s | Speed Score: 0.260 | Recent Speed: 2.76 m/s | Fall: False | Boundary: False | Step Time: 46.006s
[Gen 02] Best Gen Score: 68.914 | Best Global: 77.802 | Lap: 92.3% | Mean Speed: 2.37 m/s | Speed Score: 0.119 | Recent Speed: 2.79 m/s | Fall: False | Boundary: False | Step Time: 45.645s
[Gen 03] Best Gen Score: 47.614 | Best Global: 77.802 | Lap: 63.7% | Mean Speed: 1.63 m/s | Speed Score: 0.000 | Recent Speed: 1.67 m/s | Fall: Fa

In [7]:
# 2. Evaluate performance and generate the run video
!python run_track_bonus.py \
  --checkpoint-dir artifacts/low_level_train/best_checkpoint \
  --planner-config artifacts/highlevel_train/final_submission/planner_config.json \
  --output-dir track_eval_mlp \
  --duration-seconds 80.0 \
  --render-every 10 \
  --render-fps 10

{
  "output_dir": "/content/go2_track_bonus_repo/track_eval_mlp",
  "metrics": {
    "lap_completion": 1.0,
    "valid_distance_m": 200.0,
    "finish_time": 63.02,
    "mean_progress_speed": 3.17359568390987,
    "alive_time": 63.02,
    "fall": false,
    "fall_step": null,
    "boundary_violation": false,
    "boundary_violation_step": null,
    "rms_lateral_error": 0.7600571923196305,
    "max_lateral_error": 1.290509697842433,
    "min_boundary_margin_m": 0.7094903021575669,
    "energy_proxy": 37.54987335205078,
    "foot_slip_proxy": 3.246086835861206,
    "total_time": 80.0,
    "num_steps": 4000
  },
  "scores": {
    "completion_score": 1.0,
    "speed_score": 1.0,
    "line_keeping_score": 0.5797732439351746,
    "stability_score": 1.0,
    "efficiency_score": 0.12081286456133868,
    "composite_score": 0.871995292015102
  }
}


In [ ]:
import json
from pathlib import Path

results_path = Path("track_eval_mlp/results.json")
if results_path.exists():
    data = json.loads(results_path.read_text())
    print("--- EVALUATION METRICS ---")
    print(f"Composite Score: {data['scores']['composite_score']:.4f}")
    print(f"Lap Completion: {data['metrics']['lap_completion'] * 100:.1f}%")
    print(f"Valid Distance Covered: {data['metrics']['valid_distance_m']:.2f} meters")
    print(f"Finish Time: {data['metrics']['finish_time'] if data['metrics']['finish_time'] else 'DNF'} seconds")
    print(f"Falls Counter: {data['metrics']['fall']}")
    print(f"Boundary Violations: {data['metrics']['boundary_violation']}")
else:
    print("No evaluation output directory found. Make sure step 2 completed successfully.")

--- EVALUATION METRICS ---
Composite Score: 0.3696
Lap Completion: 58.7%
Valid Distance Covered: 117.44 meters
Finish Time: DNF seconds
Falls Counter: True
Boundary Violations: False


## 12. Create submission metadata


In [ ]:
import json
submission = {
    "team_name": TEAM_NAME,
    "track2_option": "leaderboard",
    "checkpoint_dir": "best_checkpoint",
    "planner_config": "planner_config.json",
    "planner_code": "track_bonus/planner.py",
    "planner_weights": "planner_weights.npz, if used",
    "high_level_planner_type": "learned",
    "track_eval": "track_eval/results.json",
    "notes": "Briefly describe your low-level training, learned high-level planner, and failed ideas."
}
(COURSE_REPO_DIR / "submission.json").write_text(json.dumps(submission, indent=2))
print((COURSE_REPO_DIR / "submission.json").read_text())


{
  "team_name": "change_me",
  "track2_option": "leaderboard",
  "checkpoint_dir": "best_checkpoint",
  "planner_config": "planner_config.json",
  "planner_code": "track_bonus/planner.py",
  "planner_weights": "planner_weights.npz, if used",
  "high_level_planner_type": "learned",
  "track_eval": "track_eval/results.json",
  "notes": "Briefly describe your low-level training, learned high-level planner, and failed ideas."
}


In [ ]:
import json
import shutil
from pathlib import Path

COURSE_REPO_DIR = Path("/content/go2_track_bonus_repo")

TEAM_NAME = 666  # keep your existing variable

selected_config = COURSE_REPO_DIR / "artifacts/highlevel_sweeps/robust_38/top_results/top_1_planner_config.json"
selected_eval = COURSE_REPO_DIR / "track_eval_sweeps/global_top/global_01_robust_38_top_1/seed_20260527/results.json"

# Copy selected planner to submission-friendly root files.
config_payload = json.loads(selected_config.read_text())
selected_weights = selected_config.parent / config_payload["weights_path"]

submission_config = COURSE_REPO_DIR / "planner_config.json"
submission_weights = COURSE_REPO_DIR / "planner_weights.npz"

config_payload["weights_path"] = "planner_weights.npz"
submission_config.write_text(json.dumps(config_payload, indent=2))
shutil.copy2(selected_weights, submission_weights)

eval_payload = json.loads(selected_eval.read_text())
metrics = eval_payload["metrics"]
scores = eval_payload["scores"]

submission = {
    "team_name": TEAM_NAME,
    "track2_option": "leaderboard",
    "checkpoint_dir": "best_checkpoint",
    "planner_config": "planner_config.json",
    "planner_code": "track_bonus/planner.py",
    "planner_weights": "planner_weights.npz",
    "high_level_planner_type": "learned_mlp",
    "track_eval": "track_eval/results.json",
    "selected_training_candidate": {
        "setting": "robust_38",
        "global_training_rank": 1,
        "candidate": "top_1",
    },
    "official_eval_summary": {
        "composite_score": scores["composite_score"],
        "lap_completion": metrics["lap_completion"],
        "finish_time": metrics["finish_time"],
        "mean_progress_speed": metrics["mean_progress_speed"],
        "fall": metrics["fall"],
        "boundary_violation": metrics["boundary_violation"],
        "rms_lateral_error": metrics["rms_lateral_error"],
        "max_lateral_error": metrics["max_lateral_error"],
    },
    "notes": (
        "Low-level controller is a trained Go2 PPO locomotion checkpoint. "
        "High-level planner is a learned MLP that maps official track observations "
        "to velocity/yaw commands. The MLP was warm-started by teacher distillation "
        "and refined with vectorized CEM. Final selection used an official-evaluation "
        "sweep over global top training candidates. More aggressive high-level settings "
        "were rejected because they often caused early falls or poor robustness."
    ),
}

(COURSE_REPO_DIR / "submission.json").write_text(json.dumps(submission, indent=2))
print((COURSE_REPO_DIR / "submission.json").read_text())

{
  "team_name": 666,
  "track2_option": "leaderboard",
  "checkpoint_dir": "best_checkpoint",
  "planner_config": "planner_config.json",
  "planner_code": "track_bonus/planner.py",
  "planner_weights": "planner_weights.npz",
  "high_level_planner_type": "learned_mlp",
  "track_eval": "track_eval/results.json",
  "selected_training_candidate": {
    "setting": "robust_38",
    "global_training_rank": 1,
    "candidate": "top_1"
  },
  "official_eval_summary": {
    "composite_score": 0.8916277857405706,
    "lap_completion": 1.0,
    "finish_time": 63.74,
    "mean_progress_speed": 3.1377470975839348,
    "fall": false,
    "boundary_violation": false,
    "rms_lateral_error": 0.6842076355553741,
    "max_lateral_error": 1.1113929748535156
  },
  "notes": "Low-level controller is a trained Go2 PPO locomotion checkpoint. High-level planner is a learned MLP that maps official track observations to velocity/yaw commands. The MLP was warm-started by teacher distillation and refined with ve

## 13. Final local checklist

This only checks local paths. If `track_eval/results.json` is missing, run the full evaluation command in Step 10.


In [ ]:
from pathlib import Path
expected = {
    "checkpoint": CHECKPOINT_DIR,
    "planner_config": PLANNER_CONFIG,
    "submission_json": COURSE_REPO_DIR / "submission.json",
    "track_eval_results": TRACK_EVAL_DIR / "results.json",
}
for label, path in expected.items():
    print(label, path, "OK" if path.exists() else "MISSING")
print("Submit best_checkpoint/, planner_config.json, planner weights if used, changed planner code if any, submission.json, track_eval/results.json, and your short report.")


## Lsat Part: Save

1. Save artifacts folder to drive
2. Push codes to Git


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/Proj_T2/1111/'
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Create {save_path}!")
else:
    print(f"{save_path} exist")

!cp -r /content/go2_track_bonus_repo/track_bonus/* $save_path
!cp -r /content/go2_track_bonus_repo/track_eval_mlp/* $save_path

print("Saved! You can close Colab now.")

Mounted at /content/drive
Create /content/drive/MyDrive/Proj_T2/1111/!
cp: cannot stat '/content/go2_track_bonus_repo/track_eval_mlp/*': No such file or directory
Saved! You can close Colab now.


In [ ]:
from google.colab import drive
from pathlib import Path
import json
import shutil

drive.mount("/content/drive")

repo = Path("/content/go2_track_bonus_repo")
save_path = Path("/content/drive/MyDrive/Proj_T2/0609_3")
save_path.mkdir(parents=True, exist_ok=True)

# Save global top 1-3. If you only want rank 1 and rank 3, change to [1, 3].
SAVE_RANKS = [1,2,3]

candidates_path = repo / "artifacts/highlevel_sweeps/global_training_top_candidates.json"
if not candidates_path.exists():
    raise FileNotFoundError(f"Missing: {candidates_path}")

candidates = json.loads(candidates_path.read_text())
selected = [c for c in candidates if int(c["global_training_rank"]) in SAVE_RANKS]

if not selected:
    raise RuntimeError("No selected global candidates found.")

for c in selected:
    rank = int(c["global_training_rank"])
    setting = c["setting"]
    source_candidate = c["candidate_name"]

    src_config = Path(c["planner_config"])
    if not src_config.is_absolute():
        src_config = repo / src_config

    config_payload = json.loads(src_config.read_text())
    src_weights = src_config.parent / config_payload["weights_path"]

    if not src_weights.exists():
        raise FileNotFoundError(f"Missing weights: {src_weights}")

    dest_dir = save_path / f"global_rank_{rank:02d}_{setting}_{source_candidate}"
    dest_dir.mkdir(parents=True, exist_ok=True)

    # Save portable final pair.
    new_config = dict(config_payload)
    new_config["weights_path"] = "planner_weights.npz"
    (dest_dir / "planner_config.json").write_text(json.dumps(new_config, indent=2))
    shutil.copy2(src_weights, dest_dir / "planner_weights.npz")

    # Save training metadata.
    (dest_dir / "training_candidate_record.json").write_text(json.dumps(c, indent=2))

    # Save official evaluation folders for this global candidate.
    eval_root = repo / "track_eval_sweeps/global_top"
    pattern = f"global_{rank:02d}_{setting}_{source_candidate}"
    src_eval_dir = eval_root / pattern
    if src_eval_dir.exists():
        shutil.copytree(src_eval_dir, dest_dir / "official_eval", dirs_exist_ok=True)
    else:
        print(f"Warning: eval folder not found: {src_eval_dir}")

# Save sweep summaries.
summary_files = [
    repo / "artifacts/highlevel_sweeps/global_training_top_candidates.json",
    repo / "artifacts/highlevel_sweeps/sweep_summary.csv",
    repo / "artifacts/highlevel_sweeps/sweep_summary.json",
    repo / "artifacts/highlevel_sweeps/robust_summary.csv",
    repo / "artifacts/highlevel_sweeps/robust_summary.json",
    repo / "artifacts/highlevel_sweeps/sweep_run_config.json",
]

summary_dir = save_path / "summaries"
summary_dir.mkdir(parents=True, exist_ok=True)

for p in summary_files:
    if p.exists():
        shutil.copy2(p, summary_dir / p.name)

print(f"Saved global ranks {SAVE_RANKS} to: {save_path}")
print("Each global_rank_* folder contains planner_config.json + planner_weights.npz.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved global ranks [1, 2, 3] to: /content/drive/MyDrive/Proj_T2/0609_3
Each global_rank_* folder contains planner_config.json + planner_weights.npz.


In [ ]:
drive_save = Path("/content/drive/MyDrive/Proj_T2/final_submission")
drive_save.mkdir(parents=True, exist_ok=True)

for p in [
    COURSE_REPO_DIR / "submission.json",
    COURSE_REPO_DIR / "planner_config.json",
    COURSE_REPO_DIR / "planner_weights.npz",
]:
    shutil.copy2(p, drive_save / p.name)

print(f"Backed up to {drive_save}")

Backed up to /content/drive/MyDrive/Proj_T2/final_submission
